In [ ]:
import importlib.metadata as importlib_metadata
import inspect
import logging
import os

import huggingface_hub
import torch

os.environ["CC"] = "/usr/bin/gcc"
os.environ["CXX"] = "/usr/bin/g++"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def import_vllm_or_raise():
    try:
        from vllm import LLM, SamplingParams
        return LLM, SamplingParams
    except ImportError as error:
        torch_version = importlib_metadata.version("torch")
        try:
            vllm_version = importlib_metadata.version("vllm")
        except importlib_metadata.PackageNotFoundError:
            vllm_version = "not installed"

        raise RuntimeError(
            "vLLM import failed because the installed vLLM extension is not ABI-compatible with the current PyTorch build.\n"
            f"- torch: {torch_version}\n"
            f"- torch CUDA: {torch.version.cuda}\n"
            f"- vllm: {vllm_version}\n"
            f"- original error: {error}\n\n"
            "This environment currently has torch 2.10.0+cu126, and older vLLM wheels such as 0.2.5 are not compatible with that ABI. "
            "Reinstall a vLLM build that matches your current torch/CUDA stack, or recreate the environment with a compatible torch+vllm pair."
        ) from error

LLM, SamplingParams = import_vllm_or_raise()

def maybe_hf_login() -> None:
    token = (
        os.getenv("HF_TOKEN")
        or os.getenv("HUGGINGFACE_TOKEN")
        or os.getenv("HUGGINGFACE_HUB_TOKEN")
    )
    if not token:
        return

    if not str(token).startswith("hf_"):
        logging.warning("Ignoring Hugging Face token from env because it does not look like an access token.")
        return

    login_kwargs = {
        "token": token,
        "add_to_git_credential": False,
    }
    if "skip_if_logged_in" in inspect.signature(huggingface_hub.login).parameters:
        login_kwargs["skip_if_logged_in"] = True

    huggingface_hub.login(**login_kwargs)

def ensure_cuda_available() -> None:
    cuda_available = torch.cuda.is_available()
    device_count = torch.cuda.device_count()
    if cuda_available and device_count > 0:
        return

    raise RuntimeError(
        "vLLM requires an NVIDIA GPU visible to this notebook kernel, but no CUDA device is currently available.\n"
        f"- torch.cuda.is_available(): {cuda_available}\n"
        f"- torch.cuda.device_count(): {device_count}\n\n"
        "Check that you selected the vLLM kernel on the GPU machine, the NVIDIA driver is installed, and CUDA_VISIBLE_DEVICES is not empty."
    )

maybe_hf_login()
ensure_cuda_available()
tensor_parallel_size = 2

/home/mobile/jaeyoung/all-in-one-llm-book/.venv-vllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from vllm.lora.request import LoRARequest
from huggingface_hub import snapshot_download

model_id = "allganize/Llama-3-Alpha-Ko-8B-Instruct"

llm = LLM(
    model=model_id,
    dtype="float16",
    tensor_parallel_size=tensor_parallel_size,
    gpu_memory_utilization=0.88,
    max_model_len=512,
    enforce_eager=True,
    disable_custom_all_reduce=True,
    attention_config={"backend": "TRITON_ATTN"},
    enable_lora=True,
    max_lora_rank=256,
)

INFO 03-31 13:35:58 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 512, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'enable_lora': True, 'max_lora_rank': 256, 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, use_cudnn_prefill=False, use_trtllm_ragged_deepseek_prefill=False, use_trtllm_attention=None, disable_flashinfer_prefill=True, disable_flashinfer_q_quantization=False, use_prefill_query_quantization=False), 'model': 'allganize/Llama-3-Alpha-Ko-8B-Instruct'}
INFO 03-31 13:36:00 [model.py:533] Resolved architecture: LlamaForCausalLM
WARNING 03-31 13:36:00 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 03-31 13:36:00 [model.py:1582] Using max model l

(Worker pid=26343) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=26344) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(Worker pid=26344) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
(Worker pid=26343) <frozen importlib._bootstrap_external>:1184: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


(Worker pid=26343) INFO 03-31 13:36:13 [pynccl.py:111] vLLM is using nccl==2.27.5
NCCL version 2.27.5+cuda12.9
(Worker pid=26343) WARNING 03-31 13:36:14 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=26344) WARNING 03-31 13:36:14 [symm_mem.py:67] SymmMemCommunicator: Device capability 7.5 not supported, communicator is not available.
(Worker pid=26344) INFO 03-31 13:36:14 [parallel_state.py:1717] rank 1 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 1, EP rank N/A, EPLB rank N/A
(Worker pid=26343) INFO 03-31 13:36:14 [parallel_state.py:1717] rank 0 in world size 2 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(Worker_TP0 pid=26343) INFO 03-31 13:36:14 [gpu_model_runner.py:4481] Starting to load model allganize/Llama-3-Alpha-Ko-8B-Instruct...
(Worker_TP0 pid=26343) INFO 03-31 13:36:15 [cuda.py:257] Using AttentionBackendEnum.TRITON_ATTN backend.
(Work

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:02<00:07,  2.43s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:05<00:05,  2.60s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:07<00:02,  2.65s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:08<00:00,  1.90s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:08<00:00,  2.15s/it]
(Worker_TP0 pid=26343) 


(Worker_TP0 pid=26343) INFO 03-31 13:47:52 [default_loader.py:384] Loading weights took 8.63 seconds
(Worker_TP0 pid=26343) INFO 03-31 13:47:52 [punica_selector.py:20] Using PunicaWrapperGPU.
(Worker_TP0 pid=26343) INFO 03-31 13:47:53 [gpu_model_runner.py:4566] Model loading took 8.45 GiB memory and 697.613489 seconds
(Worker_TP1 pid=26344) INFO 03-31 13:47:53 [punica_selector.py:20] Using PunicaWrapperGPU.
(Worker_TP0 pid=26343) INFO 03-31 13:48:09 [gpu_worker.py:456] Available KV cache memory: 0.61 GiB
(EngineCore pid=26279) INFO 03-31 13:48:09 [kv_cache_utils.py:1316] GPU KV cache size: 10,064 tokens
(EngineCore pid=26279) INFO 03-31 13:48:09 [kv_cache_utils.py:1321] Maximum concurrency for 512 tokens per request: 19.66x
(EngineCore pid=26279) INFO 03-31 13:48:09 [core.py:281] init engine (profile, create kv cache, warmup model) took 15.89 seconds
(EngineCore pid=26279) WARNING 03-31 13:48:12 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the perf

In [3]:
sampling_params_lora1 = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=50)
lora_adapter1 = "daje/chapter5_psychological_chatbots"
lora_adapter1_path = snapshot_download(repo_id=lora_adapter1)
lora1 = LoRARequest("lora1", 1, lora_adapter1_path)

Fetching 8 files: 100%|██████████| 8/8 [01:00<00:00,  7.59s/it]


In [4]:
sampling_params_lora2 = SamplingParams(temperature=0.1, max_tokens=50)
lora_adapter2 = "daje/chapter5_code-llama3-8B-text-to-sql-ver0.1"
lora_adapter2_path = snapshot_download(repo_id=lora_adapter2)
lora2 = LoRARequest("lora2", 2, lora_adapter2_path)

Fetching 7 files: 100%|██████████| 7/7 [00:43<00:00,  6.19s/it]


In [5]:
prompts_lora1 = [
    "일요일인데 새벽6시에 일어났어 ㅜㅜ",
    "요즘 대상포진이 걸려서 고생했어",
]

outputs = llm.generate(prompts_lora1, sampling_params_lora1, lora_request=lora1)

for output in outputs:
    generated_text = output.outputs[0].text
    print(generated_text)
    print('------')

Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

WARNING 03-31 14:06:18 [input_processor.py:141] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

(Worker_TP0 pid=26343) WARNING 03-31 14:06:22 [utils.py:268] Using default LoRA kernel configs
(Worker_TP1 pid=26344) WARNING 03-31 14:06:22 [utils.py:268] Using default LoRA kernel configs


Processed prompts: 100%|██████████| 2/2 [04:42<00:00, 141.13s/it, est. speed input: 0.10 toks/s, output: 0.35 toks/s]

. 요즘은 일도 많고, 주말에는 집에서 쉬고 싶은데 잠도 못자고 힘들어져서 더욱 스트레스 받고 있어요. 이러면 안되는데 어떻게 해야할까요?assistant
------
. 그래서 학교를 가기 싫어져서 학교를 안 가고 있는데, 이게 무슨 일인지 모르겠어. 학교에 가면 뭐가 될지, 또 무슨 일이 생길지 불안해져서 학교를 가지
------


In [6]:
prompts_lora2 = [
    """Task:최고 총액을 말해줘.'
SQL table: CREATE TABLE table_12014 (
    "Rider" text,
    "Horse" text,
    "Faults" text,
    "Round 1 + 2A Points" text,
    "Total" real
)
SQL query:""",
    "sql로 평균 구하는거 알려줘.",
]

outputs = llm.generate(prompts_lora2, sampling_params_lora2, lora_request=lora2)

for output in outputs:
    generated_text = output.outputs[0].text
    print(generated_text)
    print('------')

Processed prompts: 100%|██████████| 2/2 [04:33<00:00, 136.61s/it, est. speed input: 0.28 toks/s, output: 0.24 toks/s]

assistant

SELECT MAX("Total") FROM table_12014
------
 그리고 이름을 내림차순으로 정렬해줘.
SQL table: CREATE TABLE table_203_203 (
    id number,
    "rank" number,
    "name" text,
    "nationality" text,
    "
------
